# Urban Connectivity Atlas — Métropole du Grand Nancy

Ecological connectivity analysis per functional guild for the Grand Nancy metropolitan area.
Pipeline: ESA WorldCover + OSM land cover -> MSPA habitat morphology -> Gabriel graph ->
least-cost-path corridors -> connectivity metrics (PC, dPC, EBC, current-flow pinch points)
-> management segments.

- **Inputs**: AOI (geo.api.gouv.fr), ESA WorldCover v200, OpenStreetMap (live via osmnx).
- **Outputs**: per-guild GeoJSON layers + `stats_*.json` KPIs in `OUTPUT_DIR`.
- **Calibration**: CEREMA La Rochelle (2025); see `species_params.py`.
- **CRS**: analysis in local UTM; exact EPSG recorded in `run_metadata_{CITY}.json`.

> This notebook is duplicated per city; only `CITY` and `url_aoi` in section 0 change.
> Sections 1-4 inspect a **single** guild interactively (file saves left commented).
> Section 5 runs **all** guilds and writes the authoritative outputs.

# 0. Configuration

In [ ]:
# !pip install osmnx scikit-image seaborn
# %pip install geoai-py smoothify --target=<custom_dir> --no-cache-dir
# %pip install ipyvuetify leafmap --target=<custom_dir> --no-cache-dir

In [ ]:
import os
import sys
import json
from datetime import datetime, timezone

import numpy as np
import geopandas as gpd
import ee
import networkx as nx
import matplotlib.pyplot as plt

# --- Personal modules ---
# Flat imports matching sp_pipeline.py's own import style: this guarantees the
# notebook and the pipeline share ONE object per module (no duplicate copies,
# no reload drift between sections 1-4 and the section 5 loop).
sys.path.insert(1, "../../marion/corridor_project/modules/")
sys.path.insert(1, "../../marion/corridor_project/config/")
sys.path.insert(1, "../../Hugo/a_b_c_functions/spatial_analysis/")
import landcover as lc
import connectivity as conn
import routing as rout
import vizu_ind as viz
import species_params as spp
import sp_pipeline
from utils_polygon_smoothing import safe_smooth

# --- Google Earth Engine authentication ---
# Read credentials from environment variables; do NOT commit the key file.
service_account = os.environ.get("GEE_SERVICE_ACCOUNT", "gee-141@gee161025.iam.gserviceaccount.com")
credentials_path = os.environ.get("GEE_CREDENTIALS_PATH", "../../marion/credentials/gee161025-533af22f806b.json")
credentials = ee.ServiceAccountCredentials(service_account, credentials_path)
ee.Initialize(credentials)

# --- Case study definition (edit CITY + url_aoi when duplicating) ---
CITY = "Nancy"
url_aoi = "https://geo.api.gouv.fr/epcis?nom=Grand%20Nancy&format=geojson&geometry=contour"

# GUILD_KEY drives the single-guild inspection in sections 1-4 only;
# section 5 loops over every guild in SPECIES_CONFIG.
GUILD_KEY = "ground_mammal"
specie = spp.SPECIES_CONFIG[GUILD_KEY]

OUTPUT_DIR = f"/home/jovyan/work/team/marion/corridor_project/results_connectivity/{CITY}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

aoi_raw = gpd.read_file(url_aoi)

In [ ]:
# --- DEV ONLY: reload edited modules ---
# With unified flat imports, reload propagates to sp_pipeline (shared module objects).
# If you hit stale-state issues, restart the kernel and Run All instead.
import importlib
for _m in (lc, conn, rout, viz, spp, sp_pipeline):
    importlib.reload(_m)

In [ ]:
# Master buffer = strict AOI buffered by 2 * max(d0) over all guilds.
# Land cover is downloaded ONCE on this extent and reused for every guild.
max_d0 = max(g["graph"]["d0"] for g in spp.SPECIES_CONFIG.values())

aoi_utm, aoi_ee, utm_epsg = lc.setup_aoi(aoi_raw)
total_area_km2 = aoi_utm.area.sum() / 1e6

master_buffered_geom = gpd.GeoSeries(aoi_utm.buffer(2 * max_d0), crs=utm_epsg).to_crs(aoi_raw.crs)
master_aoi_buffered = aoi_raw.copy()
master_aoi_buffered.geometry = master_buffered_geom
master_aoi_buffered = master_aoi_buffered.dissolve()

master_aoib_utm, master_aoib_ee, master_utmb_epsg = lc.setup_aoi(master_aoi_buffered)
master_areab_km2 = master_aoib_utm.area.sum() / 1e6

print(f"Strict AOI: {total_area_km2:.1f} km2 | Buffered ({2 * max_d0} m): {master_areab_km2:.1f} km2")

lc_wc, lc_osm = lc.download_lc_data(master_aoib_ee, master_aoib_utm, master_aoi_buffered, master_utmb_epsg)

## 0.1. Provenance & run metadata (FAIR)

Archives the exact AOI used and records source versions, CRS, areas, and package
versions to `run_metadata_{CITY}.json`. OSM is fetched live, so its only pin is the
run date captured here.

In [ ]:
# --- Provenance & run metadata (FAIR) ---
RUN_UTC = datetime.now(timezone.utc).isoformat()

# Archive the exact AOI (the source API is a live, name-based query).
aoi_raw.to_file(f"{OUTPUT_DIR}/aoi_limits_{CITY}.geojson", driver="GeoJSON")

# Capture environment package versions.
from importlib.metadata import version, PackageNotFoundError
_pkgs = ["geopandas", "numpy", "networkx", "osmnx", "rasterio", "shapely",
         "scikit-image", "earthengine-api", "geemap", "xarray", "rioxarray"]
versions = {}
for _p in _pkgs:
    try:
        versions[_p] = version(_p)
    except PackageNotFoundError:
        versions[_p] = None

run_metadata = {
    "city": CITY,
    "guilds": list(spp.SPECIES_CONFIG.keys()),
    "run_utc": RUN_UTC,
    "aoi_source_url": url_aoi,
    "aoi_file": f"aoi_limits_{CITY}.geojson",
    "aoi_crs_raw": str(aoi_raw.crs),
    "utm_epsg": utm_epsg,
    "area_strict_km2": float(total_area_km2),
    "area_buffered_km2": float(master_areab_km2),
    "max_buffer_m": int(2 * max_d0),
    "landcover_source": "ESA WorldCover v200 (ee 'ESA/WorldCover/v200')",
    "osm_source": "OpenStreetMap via osmnx (live extract)",
    "osm_fetch_utc": RUN_UTC,
    "friction_calibration": "CEREMA La Rochelle (2025) pp.96-98; d0 Tab.8 p44",
    "package_versions": versions,
}
with open(f"{OUTPUT_DIR}/run_metadata_{CITY}.json", "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2, ensure_ascii=False)
print(f"Wrote run_metadata_{CITY}.json")

# 1. Land cover

Land-cover raster combining ESA WorldCover (10 m) and rasterised OSM infrastructure
(roads, rail, buildings, water).

In [ ]:
da_lc = lc.generate_guild_landcover(lc_wc, lc_osm, master_aoi_buffered, master_utmb_epsg, specie["habitat_codes"])
print(f"Land-cover codes present: {np.unique(da_lc.values)}")

# da_lc.fillna(0).astype("uint8").rio.to_raster(f"{OUTPUT_DIR}/landcover_{CITY}.tif")

# 2. Habitat morphology (MSPA)

Identifies morphological connectivity elements - cores (reservoirs) and small-core
stepping stones - from the binary habitat mask, using the guild's habitat codes.

In [ ]:
binary_wc = conn.get_binary_habitat(da_lc, specie["habitat_codes"])

gdf_cores, gdf_islets = conn.get_connectivity_elements(binary_wc, core_min_ha=1.0, islet_min_ha=0.1)
gdf_islets = gdf_islets[gdf_islets["class"] == "Stepping Stone (Small Core)"].copy()
print(f"{len(gdf_cores)} cores and {len(gdf_islets)} small-core stepping stones identified.")

# binary_wc.rio.to_raster(f"{OUTPUT_DIR}/binary_habitat_{CITY}.tif")

# 3. Graph connectivity analysis

## 3.1. Network modelling

The landscape is modelled as a graph: nodes are habitat patches, edges are potential
dispersal links. Link strength uses an exponential decay in the dispersal distance
$d_0$ (probability of movement between patches).

Graph choice (Gabriel) - trade-offs considered:
- **K-nearest neighbours**: forced degree, ignores absolute distance.
- **Relative neighbourhood graph**: too restrictive, removes loops.
- **Gabriel graph (adopted)**: captures the local neighbourhood, keeps redundant alternative paths.

Steps:
1. For each patch, search candidates within $2 \times d_0$ using edge-to-edge distance (not centroid-to-centroid).
2. Gabriel filter: take A-B as the diameter of a circle; if another patch C lies inside, drop edge A-B.
3. For surviving edges, compute movement probability (exponential decay) and its log-cost (for Dijkstra).

In [ ]:
df_nodes = conn.prepare_graph_nodes(gdf_cores, gdf_islets)
G = conn.build_gabriel_graph(df_nodes, specie)

In [ ]:
# Smooth node polygons AFTER graph construction (graph built on raw geometries for speed).
# safe_smooth() concatenates with ignore_index=True and may 'continue' past rejected
# geometries -> renumbered index, possibly dropped rows. Downstream code keys patch_masks
# and NBC on df_nodes.index while gdf_edges holds the original graph IDs, so a single drop
# would silently remap corridors. Pin a stable ID, realign by it, reinject raw geometry
# for any row that failed to smooth.
df_nodes["_node_uid"] = df_nodes.index
_nodes_presmooth = df_nodes.copy()

_smoothed = safe_smooth(df_nodes)
assert "_node_uid" in _smoothed.columns, "safe_smooth/geoai dropped attributes: cannot realign by ID."

_smoothed_geom = (_smoothed.dropna(subset=["geometry"])
                           .drop_duplicates("_node_uid", keep="first")
                           .set_index("_node_uid").geometry)
df_nodes = _nodes_presmooth.set_index("_node_uid")
df_nodes.loc[_smoothed_geom.index, "geometry"] = _smoothed_geom   # smoothed where available, raw otherwise
df_nodes.index.name = None
df_nodes["geometry"] = df_nodes.geometry.buffer(0)

## 3.2. Theoretical PC index

Probability of Connectivity (PC): landscape permeability for a guild - the probability
that two random points in the city fall in connected habitat.

> Edge-to-edge distance ignores intra-patch travel, which can overestimate connectivity
> (movement inside patches is treated as free).

In [ ]:
# G includes all nodes; total_area_km2 is the strict (unbuffered) AOI.
pc_value = conn.calculate_pc_index(G, total_area_km2)
print(f"Theoretical PC for {CITY}: {pc_value:.3f}")

gdf_edges = conn.graph_to_gdf_edges(G, master_utmb_epsg)
# gdf_edges.to_file(f"{OUTPUT_DIR}/edges_{CITY}.json", driver="GeoJSON")

# 4. Least-cost-path analysis

From Euclidean distance to friction-aware distance: the optimal path for the guild
between connected nodes over the resistance surface.

In [ ]:
resistance_raster = rout.create_resistance_surface(da_lc, specie["friction"])
# viz.plot_resistance_surface(resistance_raster, aoi_utm)
# resistance_raster.rio.to_raster(f"{OUTPUT_DIR}/friction_{CITY}.tif")

In [ ]:
# accumulated_cost is in cost-units (sum of friction x pixel size).
# threshold = d0 x average favourable friction.
d0 = specie["graph"]["d0"]
threshold = d0 * spp.FRICTION_AVG_FAVORABLE
gdf_lcp = rout.compute_lcp_network(gdf_edges, df_nodes, da_lc, specie["friction"], max_cost_threshold=threshold)

In [ ]:
# Dispersal surface: cumulative least-cost from habitats, clipped to the strict AOI.
disp = rout.compute_dispersal_surface(da_lc, df_nodes, specie["friction"])
disp_city = disp.rio.clip(aoi_utm.geometry, aoi_utm.crs, drop=True)
dispersal_raster = disp_city.where(np.isfinite(disp_city))   # inf (unreachable) -> NaN (transparent)
# viz.plot_dispersal_surface(disp_city, aoi_utm, df_nodes, threshold, gdf_lcp)
# dispersal_raster.rio.to_raster(f"{OUTPUT_DIR}/dispersal_{CITY}.tif")

## 4.1. Rupture zones

Failed corridors (blocked by uncrossable barriers) intersected with OSM obstacles,
clustered into discrete rupture points.

In [ ]:
# Clip corridors to the strict AOI (drop the buffer halo).
gdf_lcp_city = gdf_lcp[gdf_lcp.geometry.intersects(aoi_utm.union_all())].copy()

gdf_ruptures = conn.extract_rupture_points(gdf_lcp=gdf_lcp_city, lc_osm=lc_osm, friction_dict=specie["friction"])
gdf_failed_barrier = gdf_lcp_city[(gdf_lcp_city["status"] == "failed") &
                                  (gdf_lcp_city["fail_reason"] == "uncrossable_barrier")]
print(f"Barrier-blocked corridors: {len(gdf_failed_barrier)} | rupture points: {len(gdf_ruptures)}")
# viz.plot_connectivity_ruptures(gdf_lcp_city, aoi_utm, df_nodes)

## 4.2. Network metrics: tortuosity, PC

Comparing theoretical PC (Euclidean, from the graph) with real PC (LCP) quantifies the
connectivity loss caused by infrastructure and the urban matrix.

> Both PCs are computed over the **same node set** (isolated patches included on both
> sides), so the loss reflects friction, not an accounting asymmetry.

In [ ]:
# Split clipped corridors into failed / success.
gdf_lcp_city_failed = gdf_lcp_city[gdf_lcp_city["status"] == "failed"].copy()
gdf_lcp_city = gdf_lcp_city[gdf_lcp_city["status"] == "success"].copy()
# gdf_lcp_city_failed.to_file(f"{OUTPUT_DIR}/barriers_{CITY}.json", driver="GeoJSON")

# Tortuosity = real / theoretical distance.
gdf_lcp_city["tortuosity"] = gdf_lcp_city["real_dist"] / gdf_lcp_city["theoretical_dist"]
print(f"Tortuosity {CITY} - mean: {gdf_lcp_city['tortuosity'].mean():.3f} | "
      f"median: {gdf_lcp_city['tortuosity'].median():.3f}")

In [ ]:
# Real PC over the SAME node set as theoretical (isolated patches included on both sides).
G_pc = nx.from_pandas_edgelist(gdf_lcp, "node_1", "node_2")
G_pc.add_nodes_from(G.nodes(data=True))   # include distance-isolated patches (self-term ai^2 / A^2)
for node in G_pc.nodes():
    if node in G:
        G_pc.nodes[node].update(G.nodes[node])

pc_real, G_lcp = conn.calculate_pc_index_lcp(G_pc, total_area_km2, specie, gdf_lcp)
print(f"Theoretical PC: {pc_value:.3f} | Real PC: {pc_real:.3f} | "
      f"Connectivity loss: {((pc_value - pc_real) / pc_value) * 100:.1f}%")

## 4.3. Corridor metrics: dPC, EBC, current flow

- **dPC (flow)** - volume: size of connected habitats x dispersal probability.
- **EBC (rarity)** - structure: how often a corridor is the shortest path between patch pairs.
- **Current flow (vulnerability)** - funnels with no plan B (circuit-theory pinch points).

### 4.3.1. Corridor categories (dPC x EBC)

Cross flow (dPC) with rarity (EBC) into four management categories.

| Flow (dPC) | Rarity (EBC) | Corridor category |
| :--- | :--- | :--- |
| **High** | **High** | Ecological highway |
| **Low**  | **High** | Strategic bottleneck |
| **High** | **Low**  | Redundant mesh |
| **Low**  | **Low**  | Local link |

In [ ]:
gdf_lcp_city = conn.calculate_edge_dpc(gdf_lcp_city, G_lcp, total_area_km2, pc_real)   # flow
gdf_lcp_city = conn.calculate_edge_betweenness(gdf_lcp_city, G_lcp)                      # rarity
gdf_lcp_city = conn.classify_corridors(gdf_lcp_city, 0.75)                               # categories
# viz.plot_classified_corridors(gdf_lcp_city, df_nodes, aoi_utm)

### 4.3.2. Current flow (pinch points)

In [ ]:
gdf_lcp_city = conn.calculate_pinch_points_network(gdf_lcp_city, G_lcp)   # current-flow vulnerability
# gdf_lcp_city.to_file(f"{OUTPUT_DIR}/lcp_{CITY}.json", driver="GeoJSON")

### 4.3.3. From LCP to management segments

Erase in-habitat portions, split at intersections, aggregate metrics per segment, smooth, weld.

In [ ]:
gdf_segments_raw = conn.create_urban_planning_segments(gdf_lcp_city, df_nodes)
gdf_segments_smoothed = rout.safe_smooth_lines(gdf_segments_raw)
gdf_urbanplan_segments = conn.weld_segments(gdf_segments_smoothed)
# gdf_urbanplan_segments.to_file(f"{OUTPUT_DIR}/segments_amenagement_{CITY}.json", driver="GeoJSON")

# viz.plot_segment_metric(gdf_urbanplan_segments, df_nodes, aoi_utm,
#                         score_col="max_pinch_point", cmap_name="plasma",
#                         title="Vulnerability: pinch points", cbar_label="")

## 4.4. Node metrics: NBC

Node Betweenness Centrality - identifies habitat patches acting as hubs.
(`dPC connector` is disabled below: iterative node-removal, very expensive.)

In [ ]:
df_nodes_city = conn.calculate_node_betweenness(df_nodes, G_lcp, aoi_utm)
# df_nodes_city.to_file(f"{OUTPUT_DIR}/nodes_{CITY}.geojson", driver="GeoJSON")

# dPC connector (disabled - iterative node-removal, very expensive):
# df_nodes_connector = conn.calculate_node_dpc(G_lcp, total_area_km2, sp=specie)

# 5. Run all guilds

Runs the full pipeline for every guild and writes the authoritative outputs to `OUTPUT_DIR`.

In [ ]:
guild_list = list(spp.SPECIES_CONFIG.keys())
failures = {}

for guild_key in guild_list:
    try:
        sp_pipeline.sp_pipeline(guild_key, aoi_raw, CITY, OUTPUT_DIR, lc_wc, lc_osm)
    except Exception as e:
        failures[guild_key] = repr(e)
        print(f"\n[FAILED] {guild_key}: {e!r}")

if failures:
    print(f"\n{len(failures)} guild(s) failed: {list(failures)}")
else:
    print(f"\nAll {len(guild_list)} guilds completed for {CITY}.")

## 5.1. Outputs & data dictionary

Per guild, written to `OUTPUT_DIR/{guild_key}/`:
- `landcover_*.tif`, `binary_habitat_*.tif`, `friction_*.tif`, `dispersal_*.tif`
- `edges_*.json` - theoretical graph edges
- `lcp_*.json` - corridors; columns: `dPC_val`, `dPC_relative`, `ebc_score`, `pinch_point_score`, `tortuosity`, `category`, `real_dist`, `theoretical_dist`, `accumulated_cost`
- `barriers_*.json` - failed corridors; `segments_amenagement_*.json` - management segments
- `nodes_*.geojson` - patches with `nbc_score`, `total_area_ha`
- `ruptures_*.json` - rupture points
- `stats_*.json` - KPIs: `pc_theory`, `pc_real`, `connectivity_loss_pct`, `mean/median_tortuosity`, counts

Run-level: `aoi_limits_{CITY}.geojson`, `run_metadata_{CITY}.json`.

Land-cover codes: 10 Forest, 20 Shrub, 30 Grassland, 40 Cropland, 50 Diffuse urban,
51 Buildings, 52 Motorway+rail, 53 Minor roads, 54 Paths, 55 Railway, 60 Bare soil,
80 Water, 90 Wetland, 95 Mangrove. (Authoritative map: `species_params.LC_MAP`.)